## Forecasting Modeling

In [5]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [6]:
# Set constant variables
SEED   = 42
DIR    = Path('../data/raw')
np.random.seed(SEED)

# Set timestamp
TRAIN_END   = pd.Timestamp('2022-12-31')
TEST_START  = pd.Timestamp('2023-01-01')
TEST_END    = pd.Timestamp('2024-07-01')

# Validation mirrors test: last 18 months of training
VAL_START   = pd.Timestamp('2021-07-01')
VAL_END     = TRAIN_END

In [9]:
# Load dataset
df_sales = pd.read_csv(DIR / 'sales.csv', parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
df_promotions = pd.read_csv(DIR / 'promotions.csv', parse_dates=['start_date', 'end_date'])
df_web_traffic = pd.read_csv(DIR / 'web_traffic.csv', parse_dates=['date']).rename(columns={'date': 'Date'})
df_sample_submission = pd.read_csv(DIR / 'sample_submission.csv', parse_dates=['Date'])
df_order_items = pd.read_csv(DIR / 'order_items.csv', low_memory=False)
df_inventory = pd.read_csv(DIR / 'inventory.csv')

print(f"  Sales: {df_sales['Date'].min().date()} → {df_sales['Date'].max().date()} ({len(df_sales)} days)")
print(f"  Test : {TEST_START.date()} → {TEST_END.date()} ({len(df_sample_submission)} days)")
print(f"  Val  : {VAL_START.date()} → {VAL_END.date()} (mirrors test horizon)")

  Sales: 2012-07-04 → 2022-12-31 (3833 days)
  Test : 2023-01-01 → 2024-07-01 (548 days)
  Val  : 2021-07-01 → 2022-12-31 (mirrors test horizon)


In [10]:
# Build master daily dataframe
df_tmp = df_sales.copy()
df_tmp['has_promo'] = 0

for _, row in df_promotions.iterrows():
    mask = (
        (df_tmp['Date'] >= row['start_date']) &
        (df_tmp['Date'] <= row['end_date'])
    )
    df_tmp.loc[mask, 'has_promo'] = 1

df_web_traffic_copy = df_web_traffic.copy()
df_web_traffic_copy = df_web_traffic_copy.rename(columns={'date': 'Date'})
df_master = df_tmp.merge(df_web_traffic_copy[['Date', 'sessions']], on='Date')
df_master.head()

,Date,Revenue,COGS,has_promo,sessions
0,2013-01-01,5304546.99,4156070.20,0,9760
1,2013-01-02,1606940.44,1237497.84,0,10456
2,2013-01-03,2281680.01,1832133.02,0,10076
3,2013-01-04,2376895.46,1940747.07,0,9973
4,2013-01-05,2509462.77,1977027.71,0,10223


In [11]:
SPIKE_PERCENTILE = 95
spike_thresh = np.percentile(df_master['Revenue'], SPIKE_PERCENTILE)
spike_mask   = df_master['Revenue'] > spike_thresh
n_spikes     = spike_mask.sum()

print(f"  Spike threshold (p{SPIKE_PERCENTILE}): {spike_thresh:,.0f}")
print(f"  Spike days in train: {n_spikes} ({n_spikes/len(df_master)*100:.1f}%)")
print(f"  Avg Revenue (spike days): {df_master.loc[spike_mask, 'Revenue'].mean():,.0f}")
print(f"  Avg Revenue (normal days): {df_master.loc[~spike_mask, 'Revenue'].mean():,.0f}")

  Spike threshold (p95): 9,468,546
  Spike days in train: 183 (5.0%)
  Avg Revenue (spike days): 12,088,385
  Avg Revenue (normal days): 3,884,925


In [12]:
# Feature engineering
def build_features(df):
    df_trans = df.copy()
    
    # Date & Time features
    df_time = df.copy()
    df_trans['year'] = df_trans['Date'].dt.year
    df_trans['month'] = df_trans['Date'].dt.month           
    df_trans['day_of_month'] = df_trans['Date'].dt.day      
    df_trans['day_of_week'] = df_trans['Date'].dt.dayofweek 
    df_trans['day_of_year'] = df_trans['Date'].dt.dayofyear
    
    df_trans['is_weekend'] = df_trans['day_of_week'].isin([5, 6]).astype(int) # Saturdays, Sundays
    df_trans['is_month_end'] = (df_trans['day_of_month'] >= 27).astype(int)
    dist_to_25th = np.where(df_trans['day_of_month'] <= 25, 25 - df_trans['day_of_month'], 999)
    days_in_month = df_trans['Date'].dt.daysinmonth
    dist_to_1st = days_in_month - df_trans['day_of_month'] + 1
    df_trans['distance_to_payday'] = np.minimum(dist_to_25th, dist_to_1st)
    
    df_trans['is_even_year'] = (df_trans['year'] % 2 == 0).astype(int)
    df_trans['is_2019_crash'] = (df_trans['year'] == 2019).astype(int)

    # Double day
    df_trans['is_double_day'] = ((df_trans['month'] == df_trans['day_of_month']) & (df_trans['month'] >= 9)).astype(int)
    df_trans['current_double_day'] = pd.to_datetime(df_trans['year'].astype(str) + '-' + 
                                                    df_trans['month'].astype(str) + '-' + 
                                                    df_trans['month'].astype(str), errors='coerce')
    df_trans['days_to_sale'] = (df_trans['current_double_day'] - df_trans['Date']).dt.days
    df_trans['days_until_double_day'] = df_trans['days_to_sale'].apply(lambda x: x if 0 < x <= 5 else 0)
    df_trans.drop(columns=['current_double_day', 'days_to_sale'], inplace=True)
    
    # Holiday event
    df_trans['is_holiday'] = (
        ((df_trans['day_of_month'] == 30) & (df_trans['month'] == 4)) |  # 30/4
        ((df_trans['day_of_month'] == 1) & (df_trans['month'] == 5)) |   # 1/5
        ((df_trans['day_of_month'] == 2) & (df_trans['month'] == 9)) |   # 2/9
        ((df_trans['day_of_month'] == 1) & (df_trans['month'] == 1)) |   # Tet
        ((df_trans['day_of_month'] == 24) & (df_trans['month'] == 12))   # Noel
    ).astype(int)

    # Lunar New Year
    tet_dates = pd.to_datetime([
        '2012-01-23', '2013-02-10', '2014-01-31', '2015-02-19',
        '2016-02-08', '2017-01-28', '2018-02-16', '2019-02-05',
        '2020-01-25', '2021-02-12', '2022-02-01', '2023-01-22',
        '2024-02-10', '2025-01-29'
    ])
    df_trans['days_to_next_tet'] = 9999
    df_trans['dist_to_closest_tet'] = 9999

    for tet in tet_dates:
        diff = (tet - df_trans['Date']).dt.days
        mask_future = diff > 0
        df_trans.loc[mask_future, 'days_to_next_tet'] = np.minimum(
            df_trans.loc[mask_future, 'days_to_next_tet'], diff[mask_future]
        )
        current_min_abs = df_trans['dist_to_closest_tet'].abs()
        mask_closer = diff.abs() < current_min_abs
        df_trans.loc[mask_closer, 'dist_to_closest_tet'] = diff[mask_closer]

    df_trans['pre_tet_shopping_spike'] = df_trans['days_to_next_tet'].apply(lambda x: x if 0 < x <= 30 else 0)
    df_trans['is_tet_holiday'] = df_trans['dist_to_closest_tet'].apply(lambda x: 1 if -4 <= x <= 2 else 0)
    df_trans.drop(columns=['days_to_next_tet', 'dist_to_closest_tet'], inplace=True)

    # Fourier 
    df_trans['week_sin'] = np.sin(2 * np.pi * df_trans['day_of_week'] / 7)
    df_trans['week_cos'] = np.cos(2 * np.pi * df_trans['day_of_week'] / 7)
    df_trans['month_sin'] = np.sin(2 * np.pi * df_trans['month'] / 12)
    df_trans['month_cos'] = np.cos(2 * np.pi * df_trans['month'] / 12)
    df_trans['day_year_sin'] = np.sin(2 * np.pi * df_trans['day_of_year'] / 365.25)
    df_trans['day_year_cos'] = np.cos(2 * np.pi * df_trans['day_of_year'] / 365.25)

    # Anchor Lag & Rolling
    df_trans['lag_364'] = df_trans['Revenue'].shift(364)
    df_trans['lag_365'] = df_trans['Revenue'].shift(365)
    df_trans['lag_728'] = df_trans['Revenue'].shift(728)
    
    df_trans['rolling_7_lag_364'] = df_trans['lag_364'].rolling(window=7, min_periods=1).mean()
    df_trans['rolling_30_lag_364'] = df_trans['lag_364'].rolling(window=30, min_periods=1).mean()

    if 'sessions' in df_trans.columns:
        df_trans['sessions_lag_364'] = df_trans['sessions'].shift(364) # web traffic
        df_trans['rolling_7_sessions_lag_364'] = df_trans['sessions_lag_364'].rolling(window=7, min_periods=1).mean()

        # conversion rate
        df_trans['CR_lag_364'] = df_trans['lag_364'] / (df_trans['sessions_lag_364'] + 1)
        df_trans['roll7_CR_lag_364'] = df_trans['CR_lag_364'].rolling(window=7, min_periods=1).mean()
        df_trans.drop(columns=['CR_lag_364'], inplace=True)

        df_trans.drop(columns=['sessions'], inplace=True, errors='ignore')
    
    # Historical Promo Probability
    # by day and month
    train_mask = df_trans['year'] < 2023
    promo_matrix_date = df_trans[train_mask].groupby(
        ['month', 'day_of_month']
    )['has_promo'].mean().reset_index()
    promo_matrix_date.rename(columns={'has_promo': 'hist_promo_prob_by_date'}, inplace=True)
    df_trans = pd.merge(df_trans, promo_matrix_date, on=['month', 'day_of_month'], how='left')
    df_trans['hist_promo_prob_by_date'] = df_trans['hist_promo_prob_by_date'].fillna(0)
    # by dow
    promo_matrix_dow = df_trans[train_mask].groupby(
        ['day_of_week']
    )['has_promo'].mean().reset_index()
    promo_matrix_dow.rename(columns={'has_promo': 'hist_promo_prob_by_dow'}, inplace=True)
    df_trans = pd.merge(df_trans, promo_matrix_dow, on=['day_of_week'], how='left')
    df_trans['hist_promo_prob_by_dow'] = df_trans['hist_promo_prob_by_dow'].fillna(0)
    
    df_trans['hist_promo_prob'] = (df_trans['hist_promo_prob_by_date'] + df_trans['hist_promo_prob_by_dow']) / 2
    df_trans.drop(columns=['has_promo', 'hist_promo_prob_by_date', 'hist_promo_prob_by_dow'], inplace=True, errors='ignore')
    df_trans.drop(columns=['year', 'month', 'day_of_month', 'day_of_year'], inplace=True)
    return df_trans

In [13]:
# Apply feature engineering
train_part = df_master.copy()
test_part =  pd.DataFrame({'Date':df_sample_submission['Date'].values, 'Revenue':np.nan, 'COGS':np.nan})
df_full = pd.concat([train_part, test_part], ignore_index=True).sort_values('Date').reset_index(drop=True)

df_fe = build_features(df_full)
display(df_fe)

,Date,Revenue,COGS,day_of_week,is_weekend,is_month_end,distance_to_payday,is_even_year,is_2019_crash,is_double_day,...,day_year_cos,lag_364,lag_365,lag_728,rolling_7_lag_364,rolling_30_lag_364,sessions_lag_364,rolling_7_sessions_lag_364,roll7_CR_lag_364,hist_promo_prob
0,2013-01-01,5304546.99,4156070.20,1,0,0,24,0,0,0,...,0.999852,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.681801
1,2013-01-02,1606940.44,1237497.84,2,0,0,23,0,0,0,...,0.999408,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.635632
2,2013-01-03,2281680.01,1832133.02,3,0,0,22,0,0,0,...,0.998669,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.234674
3,2013-01-04,2376895.46,1940747.07,4,0,0,21,0,0,0,...,0.997634,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.235632
4,2013-01-05,2509462.77,1977027.71,5,1,0,20,0,0,0,...,0.996303,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.232759
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4195,2024-06-27,NaN,NaN,3,0,1,4,1,0,0,...,-0.998056,NaN,NaN,5503799.91,NaN,NaN,NaN,NaN,NaN,0.734674
4196,2024-06-28,NaN,NaN,4,0,1,3,1,0,0,...,-0.998981,NaN,NaN,5150082.22,NaN,NaN,NaN,NaN,NaN,0.735632
4197,2024-06-29,NaN,NaN,5,1,1,2,1,0,0,...,-0.999609,NaN,NaN,4591824.74,NaN,NaN,NaN,NaN,NaN,0.732759
4198,2024-06-30,NaN,NaN,6,1,1,1,1,0,0,...,-0.999942,NaN,NaN,3487232.00,NaN,NaN,NaN,NaN,NaN,0.732246


In [14]:
# Train-test preparing
EXCLUDE = ['Date', 'Revenue', 'COGS']
FEATURE_COLS = [c for c in df_fe.columns if c not in EXCLUDE]
print(f"Number of features: {len(FEATURE_COLS)}")

df_train = df_fe[df_fe['Date'] <= TRAIN_END].copy()
df_test = df_fe[df_fe['Date'] >= TEST_START].copy().drop(columns=['Revenue', 'COGS'])
print(f"Train: {df_train.shape}")
print(f"Test: {df_test.shape}")

Number of features: 26
Train: (3652, 29)
Test: (548, 27)


In [15]:
# Prophet train
prophet_train = df_train[['Date', 'Revenue', 'COGS', 
                          'hist_promo_prob', 'is_double_day', 
                          'pre_tet_shopping_spike', 'is_tet_holiday']].rename(columns={'Date': 'ds', 'Revenue': 'y'})
prophet_train.loc[prophet_train['ds'].dt.year == 2019, 'y'] = np.nan

prophet_model = Prophet(
    growth='logistic',
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.10,    # higher = more flexible trend
    seasonality_prior_scale=12.0,    # captures strong annual seasonality
    seasonality_mode='multiplicative', # spikes are multiplicative, not additive
    interval_width=0.80
)
prophet_model.add_country_holidays(country_name='VN')

prophet_model.add_regressor('hist_promo_prob')
prophet_model.add_regressor('is_double_day')
prophet_model.add_regressor('pre_tet_shopping_spike')

prophet_model.add_regressor('is_tet_holiday')

# Logistic growth needs cap/floor
cap = prophet_train['y'].max() * 1.5
floor = prophet_train['y'].min() * 0.5
prophet_train['cap']   = cap
prophet_train['floor'] = floor

prophet_model.fit(prophet_train, seed=SEED)

# Predict full horizon (train + test)
prophet_future = df_fe[['Date', 'hist_promo_prob', 'is_double_day', 'pre_tet_shopping_spike', 'is_tet_holiday']].copy()
prophet_future = prophet_future.rename(columns={'Date': 'ds'})
prophet_future['cap'] = cap
prophet_future['floor'] = floor

prophet_forecast = prophet_model.predict(prophet_future)
prophet_pred = prophet_forecast[['ds', 'yhat']].rename(columns={'ds': 'Date', 'yhat': 'prophet_pred'})
prophet_pred['prophet_pred'] = prophet_pred['prophet_pred'].clip(lower=0)

df_fe = df_fe.drop(columns=['prophet_pred', 'prophet_residual'], errors='ignore')
df_fe = df_fe.merge(prophet_pred, on='Date', how='left')
df_fe['prophet_residual'] = df_fe['Revenue'] - df_fe['prophet_pred']

train_pred = prophet_model.predict(prophet_train)
valid_mask = prophet_train['y'].notna()
y_true_valid = prophet_train.loc[valid_mask, 'y']
y_pred_valid = train_pred.loc[valid_mask, 'yhat'].clip(lower=0)

print(f"  Prophet trained. In-sample MAE: "
      f"{mean_absolute_error(y_true_valid, y_pred_valid):,.0f}")


00:25:21 - cmdstanpy - INFO - Chain [1] start processing
00:25:22 - cmdstanpy - INFO - Chain [1] done processing


  Prophet trained. In-sample MAE: 1,259,762


In [16]:
df_fe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4200 entries, 0 to 4199
Data columns (total 31 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Date                        4200 non-null   datetime64[ns]
 1   Revenue                     3652 non-null   float64       
 2   COGS                        3652 non-null   float64       
 3   day_of_week                 4200 non-null   int32         
 4   is_weekend                  4200 non-null   int64         
 5   is_month_end                4200 non-null   int64         
 6   distance_to_payday          4200 non-null   int32         
 7   is_even_year                4200 non-null   int64         
 8   is_2019_crash               4200 non-null   int64         
 9   is_double_day               4200 non-null   int64         
 10  days_until_double_day       4200 non-null   int64         
 11  is_holiday                  4200 non-null   int64       

In [17]:
df_fe

,Date,Revenue,COGS,day_of_week,is_weekend,is_month_end,distance_to_payday,is_even_year,is_2019_crash,is_double_day,...,lag_365,lag_728,rolling_7_lag_364,rolling_30_lag_364,sessions_lag_364,rolling_7_sessions_lag_364,roll7_CR_lag_364,hist_promo_prob,prophet_pred,prophet_residual
0,2013-01-01,5304546.99,4156070.20,1,0,0,24,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.681801,4.296713e+06,1.007834e+06
1,2013-01-02,1606940.44,1237497.84,2,0,0,23,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.635632,3.562176e+06,-1.955236e+06
2,2013-01-03,2281680.01,1832133.02,3,0,0,22,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.234674,2.781662e+06,-4.999818e+05
3,2013-01-04,2376895.46,1940747.07,4,0,0,21,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.235632,2.212949e+06,1.639465e+05
4,2013-01-05,2509462.77,1977027.71,5,1,0,20,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.232759,2.010688e+06,4.987744e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4195,2024-06-27,NaN,NaN,3,0,1,4,1,0,0,...,NaN,5503799.91,NaN,NaN,NaN,NaN,NaN,0.734674,3.379624e+06,NaN
4196,2024-06-28,NaN,NaN,4,0,1,3,1,0,0,...,NaN,5150082.22,NaN,NaN,NaN,NaN,NaN,0.735632,2.986844e+06,NaN
4197,2024-06-29,NaN,NaN,5,1,1,2,1,0,0,...,NaN,4591824.74,NaN,NaN,NaN,NaN,NaN,0.732759,2.804706e+06,NaN
4198,2024-06-30,NaN,NaN,6,1,1,1,1,0,0,...,NaN,3487232.00,NaN,NaN,NaN,NaN,NaN,0.732246,2.828243e+06,NaN


In [18]:
# ── LGBM params (tuned for MAE + RMSE balance)
LGBM_PARAMS_MAIN = {
    'objective': 'tweedie',      # Tweedie trị phân phối lệch phải cực tốt mà ko cần log1p
    'tweedie_variance_power': 1.5, # 1.5 là tỷ lệ vàng giữa Poisson và Gamma
    'metric': ['mae', 'rmse'],   # Đánh giá trực tiếp trên VND
    'n_estimators': 2000,
    'learning_rate': 0.03,
    'num_leaves': 63,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': SEED,
    'n_jobs': -1,
    'verbose': -1,
}
FEATURE_COLS_LGBM = FEATURE_COLS + ['prophet_pred']
FEATURE_COLS_LGBM = [c for c in FEATURE_COLS_LGBM if c in df_fe.columns]
print(f"Number of features: {len(FEATURE_COLS_LGBM)}")

cv_splits = [
    {'train_end': pd.Timestamp('2017-12-31'), 'val_start': pd.Timestamp('2018-01-01'), 'val_end': pd.Timestamp('2019-06-30')},
    {'train_end': pd.Timestamp('2019-12-31'), 'val_start': pd.Timestamp('2020-01-01'), 'val_end': pd.Timestamp('2021-06-30')},
    {'train_end': pd.Timestamp('2021-06-30'), 'val_start': pd.Timestamp('2021-07-01'), 'val_end': pd.Timestamp('2022-12-31')},  # MOST IMPORTANT: mirrors test
]

cv_results = []
df = df_fe.copy()
oof_predictions = np.zeros(len(df))
for fold_i, split in enumerate(cv_splits):
    # fold_train_mask = (df['Date'] <= split['train_end']) & df['Revenue'].notna()
    # fold_val_mask   = (df['Date'] >= split['val_start']) & (df['Date'] <= split['val_end']) & df['Revenue'].notna()

    fold_train_mask = (df['Date'] <= split['train_end']) & \
                      (df['Revenue'].notna()) & \
                      (df['Date'].dt.year != 2019)
                      
    fold_val_mask   = (df['Date'] >= split['val_start']) & \
                      (df['Date'] <= split['val_end']) & \
                      (df['Revenue'].notna())

    X_tr = df.loc[fold_train_mask, FEATURE_COLS_LGBM]
    y_tr = df.loc[fold_train_mask, 'Revenue']      # train main model on log scale
    X_va = df.loc[fold_val_mask, FEATURE_COLS_LGBM]
    y_va = df.loc[fold_val_mask, 'Revenue']           # evaluate on original scale

    
    # ── Train main model
    m_main = lgb.LGBMRegressor(**LGBM_PARAMS_MAIN)
    m_main.fit(
        X_tr, y_tr,
        eval_set=[(X_va, np.log1p(y_va))],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)]
    )

    fold_preds = m_main.predict(X_va)
    oof_predictions[fold_val_mask] = fold_preds
    
    fold_mae = mean_absolute_error(y_va, fold_preds)
    fold_rmse = np.sqrt(mean_squared_error(y_va, fold_preds))
    
    cv_results.append({'fold': fold_i + 1, 'mae': fold_mae, 'rmse': fold_rmse, 'best_iter': m_main.best_iteration_})
    
    print(f"✅ Fold {fold_i + 1} | Val Horizon: {split['val_start'].date()} to {split['val_end'].date()}")
    print(f"   -> Best Iter: {m_main.best_iteration_} | MAE: {fold_mae:,.0f} | RMSE: {fold_rmse:,.0f}\n")

# ── 5. Tổng kết điểm số CV
print("=" * 70)
avg_mae = np.mean([res['mae'] for res in cv_results])
avg_rmse = np.mean([res['rmse'] for res in cv_results])
print(f"🏆 TRUNG BÌNH CROSS-VALIDATION: MAE = {avg_mae:,.0f} | RMSE = {avg_rmse:,.0f}")
print("=" * 70)

Number of features: 27
✅ Fold 1 | Val Horizon: 2018-01-01 to 2019-06-30
   -> Best Iter: 3 | MAE: 2,133,744 | RMSE: 2,736,950

✅ Fold 2 | Val Horizon: 2020-01-01 to 2021-06-30
   -> Best Iter: 90 | MAE: 975,044 | RMSE: 1,229,509

✅ Fold 3 | Val Horizon: 2021-07-01 to 2022-12-31
   -> Best Iter: 615 | MAE: 712,967 | RMSE: 980,657

🏆 TRUNG BÌNH CROSS-VALIDATION: MAE = 1,273,918 | RMSE = 1,649,039


In [20]:
# Re-train full dataset
optimal_iters = int(cv_results[-1]['best_iter'] * 1.1)

full_train_mask = (df['Date'] >= df['Date'].min()) & \
                  (df['Date'] <= pd.Timestamp('2022-12-31')) & \
                  (df['Revenue'].notna()) & \
                  (df['Date'].dt.year != 2019)

X_full_train = df.loc[full_train_mask, FEATURE_COLS_LGBM]
y_full_train = df.loc[full_train_mask, 'Revenue']


FINAL_PARAMS = LGBM_PARAMS_MAIN.copy()
FINAL_PARAMS['n_estimators'] = optimal_iters

m_final = lgb.LGBMRegressor(**FINAL_PARAMS)
m_final.fit(X_full_train, y_full_train)

test_mask = df['Date'] >= pd.Timestamp('2023-01-01')
test_df = df[test_mask].copy()

X_test = test_df[FEATURE_COLS_LGBM]
# Predict Revenue
test_df['Revenue'] = m_final.predict(X_test)
test_df['Revenue'] = test_df['Revenue'].clip(lower=0)

# Luật: Năm chẵn COGS = 85% Revenue, Năm lẻ = 89% Revenue
test_df['is_even_year'] = (test_df['Date'].dt.year % 2 == 0).astype(int)
test_df['COGS'] = test_df['Revenue'] * np.where(test_df['is_even_year'] == 1, 0.85, 0.89)

# output
submission = test_df[['Date', 'Revenue', 'COGS']].iloc[:548]
file_name = 'submission.csv'
submission.to_csv(file_name, index=False)
display(submission.head())

,Date,Revenue,COGS
3652,2023-01-01,2.248199e+06,2.000897e+06
3653,2023-01-02,1.301831e+06,1.158629e+06
3654,2023-01-03,9.773183e+05,8.698133e+05
3655,2023-01-04,7.053305e+05,6.277442e+05
3656,2023-01-05,6.373399e+05,5.672325e+05
